In [1]:

import torch
import torch.nn as nn
from torchvision import models
import numpy as np
import torch.nn.functional as F

class ToneMappingVGG(nn.Module):
    def __init__(self, target_layers: list[str] = ["0" , "1" , "2"]):
        super().__init__()
        self.target_layers = target_layers

        vgg_model = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features
        self.model = vgg_model.eval()
        self._freeze_parameters()

        self.model_out: dict[str, torch.Tensor] = {}
        self._register_hooks()

    def _freeze_parameters(self) -> None:
        for parameter in self.model.parameters():
            parameter.requires_grad = False

    def _register_hooks(self) -> None:
        layer_dict = dict(self.model._modules)
        for layer_name in self.target_layers:
            layer = layer_dict[layer_name]
            layer.register_forward_hook(self._create_hook(layer_name))

    def _create_hook(self, layer_name: str) -> callable:
        def hook(module: nn.Module, input: torch.Tensor, output: torch.Tensor) -> None:
            self.model_out[layer_name] = output
        return hook

    def forward(self, image: torch.Tensor) -> dict[str, torch.Tensor]:
        _ = self.model(image)
        return {name: self.model_out[name] for name in self.target_layers}

class ToneMappingFeatureConstrastMaskingLoss(nn.Module):
    def __init__(self, patch_size: int = 13, eps: float = 1e-6, alpha_hdr: float = 0.5, alpha_tm: float = 1.0):
        super().__init__()
        self.eps = eps
        self.alpha_hdr = alpha_hdr
        self.alpha_tm = alpha_tm
        self.avg_pool = nn.AvgPool2d(
            kernel_size=patch_size,
            stride=1,
            padding=patch_size // 2,
            count_include_pad=False
        )

    def _calculate_feature_contrast(self, f_p: torch.Tensor):
        mu_b = self.avg_pool(f_p)
        C_p = (f_p - mu_b) / (torch.abs(mu_b) + self.eps)
        var_b = self.avg_pool(torch.pow(f_p - mu_b, 2))
        sigma_b = torch.sqrt(var_b.clamp(min=0) + self.eps)

        return C_p, mu_b, sigma_b

    def _calculate_fcm_masking(self, C_p: torch.Tensor, mu_b: torch.Tensor, sigma_b: torch.Tensor, alpha: float):
        sign_C = C_p / (torch.abs(C_p) + self.eps)
        M_s = sign_C * torch.pow(torch.abs(C_p), alpha)
        M_n = sigma_b / (torch.abs(mu_b) + self.eps)
        f_VGG_I = M_s / (1 + M_n)

        return f_VGG_I

    def forward(self, features_hdr: dict[str, torch.Tensor], features_tm: dict[str, torch.Tensor]):
        loss = 0.0
        for layer_name in features_hdr.keys():
            f_hdr = features_hdr[layer_name]
            f_tm = features_tm[layer_name]
            C_hdr, mu_b_hdr, sigma_b_hdr = self._calculate_feature_contrast(f_hdr)
            f_VGG_hdr = self._calculate_fcm_masking(C_hdr, mu_b_hdr, sigma_b_hdr, self.alpha_hdr)
            C_tm, mu_b_tm, sigma_b_tm = self._calculate_feature_contrast(f_tm)
            f_VGG_tm = self._calculate_fcm_masking(C_tm, mu_b_tm, sigma_b_tm, self.alpha_tm)
            layer_loss = F.l1_loss(f_VGG_tm, f_VGG_hdr)
            loss += layer_loss

        return loss


def ulaw(image: np.ndarray) -> np.ndarray:

    median_value = np.median(image)
    scale = 8.759 * np.power(median_value, 2.148) + 0.1494 * np.power(
        median_value, -2.067
    )
    transformed = np.log(1 + scale * image) / np.log(1 + scale)
    return transformed

In [2]:
!pip install brisque
from brisque import BRISQUE

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.3/140.3 kB 12.4 MB/s eta 0:00:00
  Created wheel for libsvm-official: filename=libsvm_official-3.36.0-cp312-cp312-linux_x86_64.whl size=124640 sha256=bc99cdeb723380ef178f917b0119599b01e6f57bc88950bc2cbee776b86f931f
  Stored in directory: /root/.cache/pip/wheels/df/65/4b/c3cdece6e5fa7eebef116be2d5a309f7ac50c90183cbe12c92
Successfully built libsvm-official


In [18]:
from torch.utils.data import DataLoader
from skimage.metrics import structural_similarity as ssim
import torch
import numpy as np

def evaluate_brisque(image: torch.Tensor) -> float:

    metric = BRISQUE(url=False)

    img_conv = image.cpu().detach().numpy()
    img_conv = np.transpose(img_conv, (1, 2, 0))

    img_process = (img_conv * 255).astype(np.uint8)
    return metric.score(img=img_process)

# def evaluate_SSIM_batch(model: nn.Module, data: DataLoader, device: str):
#     model.eval()
#     total_score = 0
#     num_batches = 0
#     with torch.no_grad():
#         for modified_img, original_img in data:
#             modified_img, original_img = modified_img.to(device), original_img.to(device)
#             generated_img = model(modified_img)

#             score = SSIM(generated_img, original_img)

#             total_score += score.item()
#             num_batches += 1
#     average_score = total_score / num_batches
#     print(f"Average SSIM: {average_score:.4f}")
#     return average_score



def evaluate_SSIM(original, generated) :
    score, dif = ssim(original, generated, full=True)
    return score

In [4]:
import torch
import torch.nn as nn


class ToneMappingEncoder(nn.Module):
    def __init__(self):
        super(ToneMappingEncoder, self).__init__()
        self.first_cov = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding="same"), nn.ReLU()
        )
        self.second_cov = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding="same"), nn.ReLU()
        )
        self.third_cov = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding="same"), nn.ReLU()
        )

    def forward(self, x):
        y = self.first_cov(x)
        y = self.second_cov(y)
        y = self.third_cov(y)
        return y


class ToneMappingDecoder(nn.Module):
    def __init__(self):
        super(ToneMappingDecoder, self).__init__()
        self.first_deconv = nn.Sequential(
            nn.ConvTranspose2d(192, 32, kernel_size=3, stride=1, padding="same"),
            nn.ReLU(),
        )
        self.second_deconv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=1, padding="same"),
            nn.ReLU(),
        )
        self.final_cov = nn.Sequential(
            nn.Conv2d(16, 3, kernel_size=3, stride=1, padding="same"), nn.Sigmoid()
        )

    def forward(self, x, e_low, e_mid, e_high):
        y = self.first_deconv(x)
        y = self.second_deconv(y)
        y = self.final_cov(y)
        return y + e_low + e_mid + e_high


class ToneMappingNetwork(nn.Module):
    def __init__(self):
        super(ToneMappingNetwork, self).__init__()
        self.encoder = ToneMappingEncoder()
        self.fusion_module_1 = nn.Conv2d(
            in_channels=64, out_channels=192, kernel_size=3, stride=1, padding="same"
        )
        self.fusion_module_2 = nn.Conv2d(
            in_channels=192, out_channels=192, kernel_size=1, stride=1, padding="same"
        )
        self.decoder = ToneMappingDecoder()

    def forward(self, e_1, e_2, e_3):
        e_1 = self.encoder(e_1)
        e_2 = self.encoder(e_2)
        e_3 = self.encoder(e_3)
        combined_encoding = torch.cat((e_1, e_2, e_3), dim=1)
        fuse_1 = self.fusion_module_1(combined_encoding)
        fuse_2 = self.fusion_module_2(fuse_1)
        decoded = self.decoder(fuse_2, e_1, e_2, e_3)
        return decoded

In [5]:
import os
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"


In [6]:
import cv2

def read_exr_file(image_path: str):
        image = cv2.imread(
            image_path,
            flags=cv2.IMREAD_ANYCOLOR | cv2.IMREAD_ANYDEPTH
        )
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image

def normalize_hdr_image(hdr_image):
    mean_intensity = hdr_image.mean()
    normalized_image = (0.5 * hdr_image) / mean_intensity
    return normalized_image


def generate_exposures(hdr_image):
    x_p = 1.21497
    log2 = torch.log(torch.tensor(2.0))

    c_start = torch.log(x_p / hdr_image.max()) / log2
    c_end = torch.log(x_p / torch.quantile(hdr_image, 0.5)) / log2

    c_mid = (c_start + c_end) / 2

    e_low = np.clip((2 ** c_start) * hdr_image, 0, 1)
    e_mid = np.clip((2 ** c_mid) * hdr_image, 0, 1)
    e_high = np.clip((2 ** c_end) * hdr_image, 0, 1)

    return e_low, e_mid, e_high

In [7]:
from datasets import Dataset
import os
class ToneMappingDataset(Dataset):
    def __init__(self, dir_path: str):
        self.dir_path = dir_path
        self.files = [f for f in os.listdir(dir_path) if f.endswith(".exr")]
        print(dir_path)
    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        base_filename = self.files[idx]
        file_path = os.path.join(self.dir_path, base_filename)

        hdr_image = read_exr_file(file_path)
        hdr_image = normalize_hdr_image(hdr_image)

        hdr_image_for_math = torch.tensor(hdr_image , dtype=torch.float32)
        e_low, e_mid, e_high = generate_exposures(hdr_image_for_math)

        low_exposure = torch.tensor(e_low, dtype=torch.float32).permute(2, 0, 1)
        mid_exposure = torch.tensor(e_mid, dtype=torch.float32).permute(2, 0, 1)
        high_exposure = torch.tensor(e_high, dtype=torch.float32).permute(2, 0, 1)
        hdr_image = torch.tensor(hdr_image, dtype=torch.float32).permute(2, 0, 1)

        return low_exposure, mid_exposure, high_exposure, hdr_image


In [8]:
from google.colab import drive
drive.mount('/content/drive')
%env INPUT_PATH=/content/drive/MyDrive/SIGK/sihdr/input/clip_97
%env REFERENCE_PATH=/content/drive/MyDrive/SIGK/sihdr/reference

input_data = ToneMappingDataset(os.getenv('INPUT_PATH'))
reference_data = ToneMappingDataset(os.getenv('REFERENCE_PATH'))
print(len(reference_data.files))

loss = ToneMappingFeatureConstrastMaskingLoss()
vgg = ToneMappingVGG()


Mounted at /content/drive
env: INPUT_PATH=/content/drive/MyDrive/SIGK/sihdr/input/clip_97
env: REFERENCE_PATH=/content/drive/MyDrive/SIGK/sihdr/reference
/content/drive/MyDrive/SIGK/sihdr/input/clip_97
/content/drive/MyDrive/SIGK/sihdr/reference
181
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 214MB/s]


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test1, test2, test3, test4 = reference_data[1]

loss.eval()
loss.to(device)
vgg.to(device)

test1 = test1.to(device)
test2 = test2.to(device)

features_tm = vgg(test1)
features_hdr = vgg(test2)
calc_loss = loss(features_hdr, features_tm)

/tmp/ipython-input-1237178203.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  low_exposure = torch.tensor(e_low, dtype=torch.float32).permute(2, 0, 1)
/tmp/ipython-input-1237178203.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mid_exposure = torch.tensor(e_mid, dtype=torch.float32).permute(2, 0, 1)
/tmp/ipython-input-1237178203.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  high_exposure = torch.tensor(e_high, dtype=torch.float32).permute(2, 0, 1)
